# NOTEBOOK3: Implementing LASSO

Generate data

In [ ]:
import pandas as pd
import numpy as np

def generate_data(n_samples, n_features, noise):
    d = {}
    f1 = np.random.randn(n_samples)
    d["f1"] = f1
    for i in range(2, n_features + 1):
        alpha_i = np.random.randn()
        d[f"f{i}"] = alpha_i * d[f"f{i-1}"] + noise
    return pd.DataFrame(d)

data = generate_data(150, 5, 0.2)
data.head()




Random forest + SHAP

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import shap

X = data.drop("f5", axis=1)
y = data["f5"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

preds = model.predict(val_X)
mae = mean_absolute_error(val_y, preds)
mae


In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(val_X)
shap.summary_plot(shap_values, val_X)
shap.summary_plot(shap_values, val_X, plot_type="bar")

decorrelated data  with f6 useless

In [ ]:
d = {}
d["f1"] = np.random.randn(150)
for i in range(2, 6):
    d[f"f{i}"] = np.random.randn(150)
d["f6"] = np.random.randn(150)
df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

preds = model.predict(val_X)
mae = mean_absolute_error(val_y, preds)
mae


In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(val_X)
shap.summary_plot(sv, val_X)
shap.summary_plot(sv, val_X, plot_type="bar")

eli5 ermutation matrix in this case

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(model, random_state=1).fit(val_X, val_y)
eli5.show_weights(perm, feature_names=val_X.columns.tolist())


LASSO L1 (Lasso) does feature selection, can set coeff exactly to 0. L2 (Ridge) shrinks coefficients but keeps them non-zero. ElasticNet balances both

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
ridge = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=1.0))])
enet = Pipeline([("scaler", StandardScaler()), ("m", ElasticNet(alpha=0.1, l1_ratio=0.5))])

lasso.fit(train_X, train_y)
ridge.fit(train_X, train_y)
enet.fit(train_X, train_y)

lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))
ridge_coefs = dict(zip(train_X.columns, ridge.named_steps["m"].coef_))
enet_coefs = dict(zip(train_X.columns, enet.named_steps["m"].coef_))

lasso_coefs, ridge_coefs, enet_coefs


almost like the permutations!

Let's try now with f6 depending almost entirely from f3

In [ ]:
d = {}
d["f1"] = np.random.randn(800)
for i in range(2, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.5*d["f1"] + 2*d["f2"] + 1000*d["f3"] - 0.1*d["f4"]
df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

shap.initjs()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(val_X)
shap.summary_plot(sv, val_X)
shap.summary_plot(sv, val_X, plot_type="bar")


In [ ]:
lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
ridge = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=1.0))])
enet = Pipeline([("scaler", StandardScaler()), ("m", ElasticNet(alpha=0.1, l1_ratio=0.5))])

lasso.fit(train_X, train_y)
ridge.fit(train_X, train_y)
enet.fit(train_X, train_y)

lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))
ridge_coefs = dict(zip(train_X.columns, ridge.named_steps["m"].coef_))
enet_coefs = dict(zip(train_X.columns, enet.named_steps["m"].coef_))

lasso_coefs, ridge_coefs, enet_coefs

the 3 methods identify the weight of the coefficients

now with a cofounding variable

In [ ]:
d = {}
d["f1"] = np.random.randn(800)
d["f2"] = d["f1"]*10 + np.random.randn(800)*0.1
for i in range(3, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.6*d["f1"] + 6*d["f2"] + 0.2*d["f3"] - 0.1*d["f4"] + 3

df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

shap.initjs()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(val_X)
shap.summary_plot(sv, val_X)
shap.summary_plot(sv, val_X, plot_type="bar")


In [ ]:
lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
ridge = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=1.0))])
enet = Pipeline([("scaler", StandardScaler()), ("m", ElasticNet(alpha=0.1, l1_ratio=0.5))])

lasso.fit(train_X, train_y)
ridge.fit(train_X, train_y)
enet.fit(train_X, train_y)

lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))
ridge_coefs = dict(zip(train_X.columns, ridge.named_steps["m"].coef_))
enet_coefs = dict(zip(train_X.columns, enet.named_steps["m"].coef_))

lasso_coefs, ridge_coefs, enet_coefs

Lasso works immensely better at this

next steps: understand the way lasso and ridge work. why is it more useful for confounding variables? mathematically why is shapley not able to identify them? what offsets the results of lasso/ridge? 